# Exercise - Forecasting with Fundamentals

# 1. Carry Strategy for Dividend Yield

In this exercise, we consider the dividend-yield of individual stocks in order to invest in **high-carry** securities.

Note that we could consider the full analysis below with other metrics, including the PE ratio provided in the same data set.

#### Data

Use the data in `data/spx_data_weekly.xlsx`.
* `spx data` - time-series data for price, dividend-price ratio, and price-earnings ratio
* `sector data` - ETFs on sub-sectors
* `additional data` - `SPY`, `SHV`, and other benchmarks
* `rates data` - index levels for rates

Risk-free rate could be 
* `SHV`
*  `USGG3M Index` - annualized percent

In [1]:
import pandas as pd

FILE_DATA = '../data/spx_data_weekly.xlsx'
SHEET_INFO = 'spx names'

info = pd.read_excel(FILE_DATA,sheet_name=SHEET_INFO)
info.set_index('ticker',inplace=True)
info.rename(columns={'security_name':'security name'},inplace=True)

display(info)

,name,gics_sector_name,mkt cap
ticker,,,
A,Agilent Technologies Inc,Health Care,4.162353e+10
AAPL,Apple Inc,Information Technology,4.025226e+12
ABBV,AbbVie Inc,Health Care,4.106695e+11
ABNB,Airbnb Inc,Consumer Discretionary,7.510938e+10
ABT,Abbott Laboratories,Health Care,2.270793e+11
...,...,...,...
XYZ,Block Inc,Financials,3.675158e+10
YUM,Yum! Brands Inc,Consumer Discretionary,4.137582e+10
ZBH,Zimmer Biomet Holdings Inc,Health Care,1.781850e+10


In [2]:
def load_ts(file_data,sheet_name):
    ts = pd.read_excel(file_data,sheet_name=sheet_name,
                    header=[0,1],                  
                    index_col=0)
                    
    ts.columns.names = ["ticker","field"]

    ts.rename(columns={
        'PX_LAST':'price',
        'EQY_DVD_YLD_IND':'dvd yld',
        'PE_RATIO':'pe ratio'
    }, level=1, inplace=True)

    return ts

In [3]:
SHEET_TS = 'spx data'
spx = load_ts(FILE_DATA,SHEET_TS)
display(spx.tail().style.format('{:.2f}',na_rep='').format_index(lambda x: x.strftime('%Y-%m-%d')))

### 1.0. Data Processing

Filter the data to drop any ticker which does not have at least `5` years of continuous price data.

In [8]:
minimum_years=5
trading_time=52
minimum_obs=minimum_years*trading_time

spx.index = pd.to_datetime(spx.index)

prices = spx.xs('price', axis=1, level=1)

good_tickers=prices.notna().sum().ge(minimum_obs)
good_tickers=good_tickers[good_tickers].index
spx_filtered=spx.loc[:, spx.columns.get_level_values(0).isin(good_tickers)].copy()
spx_filtered

ticker             A                       AAPL                       ABBV  \
field          price dvd yld pe ratio     price dvd yld pe ratio     price   
date                                                                         
2015-07-03   36.4044  1.0988  27.4720   28.2589  1.8401  13.0592   44.2648   
2015-07-10   36.2388  1.1038  27.3471   27.5527  1.8873  12.7328   44.9267   
2015-07-17   36.7447  1.0886  27.7288   28.9696  1.7950  13.3876   45.7570   
2015-07-24   36.1560  1.1063  27.2846   27.8253  1.8688  12.8588   44.5083   
2015-07-31   37.6645  1.0620  26.6192   27.1101  1.9181  12.5283   45.7701   
...              ...     ...      ...       ...     ...      ...       ...   
2025-05-30  111.4610  0.8900  22.0155  200.4280  0.5189  28.2604  183.1956   
2025-06-06  115.3251  0.8602  22.7787  203.4916  0.5111  28.6924  186.8573   
2025-06-13  116.5999  0.8508  23.0305  196.0373  0.5305  27.6413  188.0877   
2025-06-20  115.0861  0.8620  22.7315  200.5777  0.5185  28.2815  182.3983   
2025-06-27  118.6813  0.8359  23.4416  200.6576  0.5183  27.6305  179.4551   

ticker                            ABT  ...      YUM      ZBH                   \
field      dvd yld pe ratio     price  ... pe ratio    price dvd yld pe ratio   
date                                   ...                                      
2015-07-03  4.6086  11.9239   40.4951  ...  18.6689  96.5853  0.9111  18.2975   
2015-07-10  4.5407  12.1022   40.9213  ...  18.5282  95.3533  0.9229  18.0642   
2015-07-17  4.4583  12.3258   41.0942  ...  18.0008  95.9693  0.9170  18.1809   
2015-07-24  4.5834  11.9895   42.0413  ...  17.7611  95.2194  0.9242  18.0388   
2015-07-31  4.4571  12.3294   41.7448  ...  17.9762  92.9072  0.9472  17.6008   
...            ...      ...       ...  ...      ...      ...     ...      ...   
2025-05-30  3.5809  23.7228  132.3943  ...  26.0513  91.7072  1.0468  17.9771   
2025-06-06  3.5107  24.1970  132.3943  ...  26.2124  91.9161  1.0444  18.0180   
2025-06-13  3.4877  24.3563  134.4162  ...  26.0622  90.8216  1.0570  17.8035   
2025-06-20  3.5965  23.6195  131.8096  ...  25.1681  90.7619  1.0577  17.7918   
2025-06-27  3.6555  23.2384  133.1872  ...  26.8603  91.2367  1.0522  17.8848   

ticker        ZBRA                         ZTS                   
field        price dvd yld  pe ratio     price dvd yld pe ratio  
date                                                             
2015-07-03  112.43     NaN  151.5166   44.5029  0.7460  26.8168  
2015-07-10  109.75     NaN  147.9049   43.2593  0.7675  26.0674  
2015-07-17  114.18     NaN  153.8750   44.0331  0.7540  26.5337  
2015-07-24  110.89     NaN  149.4412   45.5900  0.7282  27.4718  
2015-07-31  107.63     NaN  145.0478   45.1201  0.7358  27.1887  
...            ...     ...       ...       ...     ...      ...  
2025-05-30  289.77     NaN   27.0190  167.4848  1.1941  27.9582  
2025-06-06  295.36     NaN   27.5403  169.0044  1.1834  28.2119  
2025-06-13  283.61     NaN   26.4447  163.2140  1.2254  27.2453  
2025-06-20  294.04     NaN   27.4172  156.1026  1.2812  26.0582  
2025-06-27  309.26     NaN   28.5673  155.1094  1.2894  25.8924  

[522 rows x 1455 columns]

### 1.1. Data Processing

Report the highest and lowest dividend-yielding stocks...

* for any given date across the entire panel
* taking an average of the past year

Which stocks were they? Was it driven more by changes in D or P?

In [14]:
dividends = spx_filtered.xs('dvd yld',axis=1, level=1)

max_yield = dividends.max(axis=1)
max_yield_ticker = dividends.idxmax(axis=1)

min_yield = dividends.min(axis=1)
min_yield_ticker = dividends.idxmin(axis=1)

yields_by_date = pd.DataFrame({
    'max_ticker': max_yield_ticker,
    'max_yield':  max_yield,
    'min_ticker': min_yield_ticker,
    'min_yield':  min_yield,
})
yields_by_date

,max_ticker,max_yield,min_ticker,min_yield
date,,,,
2015-07-03,DD,20.8790,CI,0.0270
2015-07-10,DD,20.8668,CI,0.0277
2015-07-17,DD,20.9647,CI,0.0284
2015-07-24,DD,23.3362,CI,0.0299
2015-07-31,DD,22.7957,CI,0.0302
...,...,...,...,...
2025-05-30,DOW,10.2396,NVDA,0.0296
2025-06-06,DOW,9.9491,NVDA,0.0282
2025-06-13,DOW,9.4999,NVDA,0.0282


In [15]:
last_date = dividends.index.max()
start_date = last_date - pd.DateOffset(years=1)

last_year_dividends = dividends.loc[start_date:last_date]
average_yield = last_year_dividends.mean(axis=0)

highest_avg_ticker = average_yield.idxmax()
lowest_avg_ticker  = average_yield.idxmin()

highest_avg_yield = average_yield.max()
lowest_avg_yield  = average_yield.min()

print("Highest avg yield:", highest_avg_ticker, highest_avg_yield)
print("Lowest  avg yield:", lowest_avg_ticker, lowest_avg_yield)

Highest avg yield: MO 7.976584905660378
Lowest  avg yield: NVDA 0.031783018867924534


### 1.2. A Carry Strategy

For this strategy, use the `dvd yld` data.

Build a portfolio where at every time $t$, you set weights, $w_t$ that invest...
* long the highest ranking `20%` stocks.

For now, go equal weights of `0.01` in any stock for which you're long or short. 

#### Realized Returns
Start at the beginning of the sample.
* At time $t$, rank the stocks. 
* The selected stocks will earn $r_{t+1}$.

In [16]:
import numpy as np

div_yld = spx_filtered.xs('dvd yld', axis=1, level=1)
prices = spx_filtered.xs('price', axis=1, level=1)

ret_fwd = prices.shift(-1) /prices - 1
div_yld = div_yld.iloc[:-1]
ret_fwd = ret_fwd.iloc[:-1]

proportion = 0.2
standard_weight=0.01

ranks = div_yld.rank(axis=1, method='first', ascending=False)
n_stocks = div_yld.notna().sum(axis=1)
cutoff = (n_stocks * proportion).astype(int)   

long_only=ranks.le(cutoff, axis=0)

weights = long_only.astype(float) *standard_weight
carry_return=(weights*ret_fwd).sum(axis=1)

carry_return



date
2015-07-03    0.004695
2015-07-10    0.004436
2015-07-17   -0.017047
2015-07-24    0.016680
2015-07-31   -0.005483
                ...   
2025-05-23    0.011326
2025-05-30    0.005368
2025-06-06    0.000485
2025-06-13   -0.002332
2025-06-20    0.008884
Length: 521, dtype: float64

### 1.3. Long-Short

Re-do the strategy of the previous section, but this time go long and short...

At any given time $t$, set the vector $w_t$ as follows...
* long the highest-ranking `20%` of stocks.
* short the lowest-ranking `20%` of stocks.
* take no position in all other stocks.

In [17]:
div_yld = spx_filtered.xs('dvd yld', axis=1, level=1)
prices = spx_filtered.xs('price', axis=1, level=1)

ret_fwd = prices.shift(-1) /prices - 1
div_yld = div_yld.iloc[:-1]
ret_fwd = ret_fwd.iloc[:-1]

proportion = 0.2
standard_weight=0.01

ranks = div_yld.rank(axis=1, method='first', ascending=False)
n_stocks = div_yld.notna().sum(axis=1)
cutoff = (n_stocks * proportion).astype(int)   

long_only=ranks.le(cutoff, axis=0)
weights_long = long_only.astype(float) *standard_weight
carry_return_long=(weights*ret_fwd).sum(axis=1)

short_only = ranks.gt(n_stocks - cutoff, axis=0)
weights_long_short = (long_only.astype(float) - short_only.astype(float)) * standard_weight

carry_return_long_short = (weights_long_short * ret_fwd).sum(axis=1)

carry_return_long_short

date
2015-07-03    0.004453
2015-07-10   -0.001672
2015-07-17   -0.006292
2015-07-24    0.005288
2015-07-31   -0.000745
                ...   
2025-05-23    0.001647
2025-05-30   -0.006956
2025-06-06    0.012479
2025-06-13   -0.003716
2025-06-20   -0.016309
Length: 521, dtype: float64

### 1.4. Performance

Calculate the return of your portfolio over time. 

Report the following annualized stats
* mean
* volatility
* Sharpe 

Also calculate the tail-risk stats
* skewness
* VaR (5th quantile)
* CVaR (5th quantile)
* max drawdown

For both tables, compare to `SPY`, found in the `additional data` tab.

In [ ]:
SHEET_BENCH = 'additional data'
bench = load_ts(FILE_DATA,SHEET_BENCH)
display(bench[['SPY']].tail().style.format('{:.2f}',na_rep='').format_index('{:%Y-%m-%d}'))

In [28]:
spy_price = bench['SPY']['price'].astype(float)   # index = date

# --- weekly SPY returns ---
spy_return = spy_price.pct_change()

# (optional) align to your strategy sample
spy_return = spy_return.loc[carry_return.index]

In [29]:
import numpy as np
import pandas as pd

# --- SPY weekly returns (adapt this line to your data structure) ---
# Example if you read "additional data" into a df called `extra`:
# spy_price = extra['SPY']
# Align with strategy index:

# --- helper: performance stats for one return series ---
def perf_stats(r, periods_per_year=52):
    r = r.dropna()
    
    mean_ann = r.mean() * periods_per_year
    vol_ann  = r.std(ddof=1) * np.sqrt(periods_per_year)
    sharpe   = mean_ann / vol_ann if vol_ann != 0 else np.nan
    
    skew     = r.skew()
    var_5    = r.quantile(0.05)                       # 5% VaR
    cvar_5   = r[r <= var_5].mean()                   # 5% CVaR
    
    # max drawdown
    cum = (1 + r).cumprod()
    peak = cum.cummax()
    drawdown = cum / peak - 1
    max_dd = drawdown.min()
    
    return pd.Series({
        'mean_ann': mean_ann,
        'vol_ann': vol_ann,
        'sharpe': sharpe,
        'skewness': skew,
        'VaR_5': var_5,
        'CVaR_5': cvar_5,
        'max_drawdown': max_dd
    })

# --- build comparison table ---
results = pd.DataFrame({
    'Carry long-only': perf_stats(carry_return),
    'Carry long-short': perf_stats(carry_return_long_short),
    'SPY': perf_stats(spy_return),
}).T

results


,mean_ann,vol_ann,sharpe,skewness,VaR_5,CVaR_5,max_drawdown
Carry long-only,0.111281,0.162669,0.684097,0.274936,-0.028637,-0.049597,-0.360059
Carry long-short,-0.005456,0.095209,-0.057307,1.088094,-0.018523,-0.027307,-0.241689
SPY,0.137625,0.173063,0.795233,-0.594387,-0.033678,-0.057658,-0.318291


***

# 2. Attribution

### 2.1. Market Exposure

For both the long-only (LO) and long-short (LS) strategies, estimate a linear factor decomposition (LFD) against `SPY`

Report the...
* alpha (annualized)
* beta
* r-squared

Also show the correlation matrix of the strategies.

How do the LO and LS strategies compare to `SPY`?

In [34]:
import statsmodels.api as sm

ret_df = pd.DataFrame({
    'LO':  carry_return,
    'LS':  carry_return_long_short,
    'SPY': spy_return,
}).dropna()

def lfd_against_spy(strategy_name, periods_per_year=52):
    y = ret_df[strategy_name]
    X = sm.add_constant(ret_df['SPY'])

    model = sm.OLS(y, X).fit()

    alpha_weekly = model.params['const']
    beta         = model.params['SPY']
    r2           = float(model.rsquared)

    alpha_annual = alpha_weekly * periods_per_year

    return pd.Series({
        'alpha_ann': alpha_annual,
        'beta':       beta,
        'r_squared':  r2
    }, name=strategy_name)

# --- 3) Run for LO and LS ---
lfd_LO = lfd_against_spy('LO')
lfd_LS = lfd_against_spy('LS')

lfd_table = pd.concat([lfd_LO, lfd_LS], axis=1).T
display(lfd_table)


,alpha_ann,beta,r_squared
LO,0.118187,-0.052035,0.003059
LS,-0.009570,0.026581,0.002331


### 2.2. Sector Regression

Estimate a multivariate LFD for both LO and LS against all the sector ETFs.
* Note that `SHV` is not a sector ETF but rather a money-market fund. Exclude it.
* Exclude `SPY`.

Report the same stats as in `2.1.`

In [35]:
SHEET_SECTOR_INFO = 'sector names'
info_sectors = pd.read_excel(FILE_DATA,sheet_name=SHEET_SECTOR_INFO).rename(columns={'Unnamed: 0':'ticker'}).set_index('ticker').drop(columns=['gics_sector_name'])
info_sectors['cur_mkt_cap'] = info_sectors['cur_mkt_cap'].astype(float) / 1e9
info_sectors.rename(columns={'cur_mkt_cap':'fund size ($ billions)'},inplace=True)
display(info_sectors.style.format({'fund size ($ billions)':'${:,.2f}'}))

,security_name,fund size ($ billions)
ticker,,
XLB US Equity,Materials Select Sector SPDR F,$5.13
XLC US Equity,Communication Services Select,$25.31
XLE US Equity,Energy Select Sector SPDR Fund,$28.20
XLF US Equity,Financial Select Sector SPDR F,$52.73
XLI US Equity,Industrial Select Sector SPDR,$23.76
XLK US Equity,Technology Select Sector SPDR,$93.26
XLP US Equity,Consumer Staples Select Sector,$15.05
XLRE US Equity,Real Estate Select Sector SPDR,$7.54
XLU US Equity,Utilities Select Sector SPDR F,$22.19


In [36]:
SHEET_SECTORS = 'sector data'
sectors = load_ts(FILE_DATA,SHEET_SECTORS)
display(sectors.tail().style.format('{:.1f}',na_rep='').format_index(lambda x: x.strftime('%Y-%m-%d')))

In [56]:
WEEKS_PER_YEAR=52

sector_prices = sectors.xs('price', axis=1, level='field').astype(float)

drop_cols = ['SPY', 'SHV']

sector_prices=sector_prices.drop(columns=drop_cols, errors='ignore')

sector_returns=sector_prices.pct_change()

reg_df = pd.concat(
    [
        carry_return.rename('LO'),
        carry_return_long_short.rename('LS'),
        sector_returns
    ],
    axis=1
)
reg_df = reg_df.replace([np.inf, -np.inf], np.nan).dropna()

y_LO = reg_df['LO']
y_LS = reg_df['LS']
X    = reg_df.drop(columns=['LO', 'LS'])

X = sm.add_constant(X)

#multivariate regressions
model_LO = sm.OLS(y_LO, X).fit()
model_LS = sm.OLS(y_LS, X).fit()

alpha_ann_LO = model_LO.params['const'] * WEEKS_PER_YEAR
alpha_ann_LS = model_LS.params['const'] * WEEKS_PER_YEAR

lfd_sector_stats = pd.DataFrame(
    {
        'alpha_ann': [alpha_ann_LO, alpha_ann_LS],
        'r_squared': [model_LO.rsquared, model_LS.rsquared],
    },
    index=['LO', 'LS']
)

betas_LO = model_LO.params.drop('const')
betas_LS = model_LS.params.drop('const')

sector_betas = pd.DataFrame({'LO_beta': betas_LO, 'LS_beta': betas_LS})

lfd_sector_stats, sector_betas

(    alpha_ann  r_squared
 LO   0.123031   0.060525
 LS  -0.012927   0.116344,
        LO_beta   LS_beta
 XLK   0.013171  0.082119
 XLI   0.026161  0.056847
 XLF  -0.024489 -0.046141
 XLC   0.013016 -0.019481
 XLRE -0.224585 -0.127511
 XLE  -0.000117  0.044236
 XLY   0.008931 -0.058372
 XLB   0.152128  0.018811
 XLV   0.014736  0.067678
 XLU  -0.095899 -0.118165
 XLP   0.031835  0.169412)

### 2.3. Sector Neutrality

Is your LO or LS implementation of the carry strategy neutral to sectors? To which sector does it have the largest exposure?

**Remember**

Unless regressors are standardized, comparing betas directly can be misleading. You might consider comparing $\beta_i\sigma_i$ when deciding which exposure is most substantial.

In [59]:
sector_sigma=sector_returns.std(ddof=1)

sector_exposure = pd.DataFrame({
    'beta_LO': sector_betas['LO_beta'],
    'beta_LS': sector_betas['LS_beta'],
    'sigma': sector_sigma
})

# beta * sigma
sector_exposure['beta_sigma_LO'] = sector_exposure['beta_LO'] * sector_exposure['sigma']
sector_exposure['beta_sigma_LS'] = sector_exposure['beta_LS'] * sector_exposure['sigma']

display(sector_exposure)

# which sector has the largest exposure
max_LO_sector = sector_exposure['beta_sigma_LO'].abs().idxmax()
max_LS_sector = sector_exposure['beta_sigma_LS'].abs().idxmax()

print("Largest LO exposure:", max_LO_sector,
      sector_exposure.loc[max_LO_sector, 'beta_sigma_LO'])

print("Largest LS exposure:", max_LS_sector,
      sector_exposure.loc[max_LS_sector, 'beta_sigma_LS'])

,beta_LO,beta_LS,sigma,beta_sigma_LO,beta_sigma_LS
XLK,0.013171,0.082119,0.030365,0.000400,0.002494
XLI,0.026161,0.056847,0.028388,0.000743,0.001614
XLF,-0.024489,-0.046141,0.031066,-0.000761,-0.001433
XLC,0.013016,-0.019481,0.028750,0.000374,-0.000560
XLRE,-0.224585,-0.127511,0.029822,-0.006698,-0.003803
XLE,-0.000117,0.044236,0.041067,-0.000005,0.001817
XLY,0.008931,-0.058372,0.029902,0.000267,-0.001745
XLB,0.152128,0.018811,0.028960,0.004406,0.000545
XLV,0.014736,0.067678,0.023308,0.000343,0.001577
XLU,-0.095899,-0.118165,0.026694,-0.002560,-0.003154


Largest LO exposure: XLRE -0.006697591513634661
Largest LS exposure: XLRE -0.003802645960300451


The startegy seems to be very sector neutral, given that the betas seem to ber very small or even negative for every sector. 

### 2.4. Magnificent Seven

Construct an equally-weighted portfolio of the following tickers. Call this `MAG`. Estimate a LFD on both `SPY` and `MAG`. Do this for both the LO and LS strategies.

Report the stats from `2.1`.

Comment on what you conclude from this regression.

In [60]:
TICKS_MAG = ['AAPL','MSFT','GOOG','AMZN','NVDA','META','TSLA']
display(TICKS_MAG)

['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA', 'META', 'TSLA']

In [62]:
all_prices = spx_filtered.xs('price', axis=1, level=1).astype(float)

mag_prices = all_prices[TICKS_MAG]

mag_index=mag_prices.mean(axis=1)

mag_return=mag_index.pct_change()

reg_df_mag = pd.DataFrame({
    'LO':  carry_return,
    'LS':  carry_return_long_short,
    'SPY': spy_return,
    'MAG': mag_return,  
}).dropna()

def linear_decomposition_mag(strategy):
    y = reg_df_mag[strategy]
    X = reg_df_mag[['SPY', 'MAG']]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    alpha_weekly = model.params['const']
    alpha_annual = alpha_weekly * WEEKS_PER_YEAR

    return pd.Series({
        'alpha_ann': alpha_annual,
        'beta_SPY':  model.params['SPY'],
        'beta_MAG':  model.params['MAG'],
        'r_squared': model.rsquared,
    }, name=strategy)

# run for LO and LS
lfd_LO_mag = linear_decomposition_mag('LO')
lfd_LS_mag = linear_decomposition_mag('LS')

lfd_mag_table = pd.concat([lfd_LO_mag, lfd_LS_mag], axis=0).to_frame().T
display(lfd_LO_mag)
display(lfd_LS_mag)

alpha_ann    0.117104
beta_SPY    -0.064839
beta_MAG     0.010827
r_squared    0.003158
Name: LO, dtype: float64

alpha_ann   -0.011310
beta_SPY     0.006014
beta_MAG     0.017391
r_squared    0.003080
Name: LS, dtype: float64

The strategy seems to be more correlated with the MAG7 rather than the entire stock exchange. However, the r-squared does not improve a lot so the SPy movements are already largely influenced by the MAG7.

***

# 3. Dynamic hedged

It's one thing to ex-post evaluate whether the strategy had alpha beyond sector exposures. That raises the question as to whether we can caputre that when we're imperfectly hedging the sector exposure in real time.

At each date we:

1. Look back over a rolling 5-year window of weekly data.
2. Regress the portfolio return on the sector ETF returns.
3. Use the estimated betas as hedge ratios going forward one step.
4. Form a sector-hedged return by combining the strategy with
   an offsetting position in the sector ETFs.

We do this for both the long-only and long-short variants.

### 3.1.

Report the univariate stats of the hedged strategies.

### 3.2. 

Report sector LFD of the sector-hedged strategies.

In [65]:
#3.1 
ROLL_YEARS = 5
ROLL_WINDOW = ROLL_YEARS * WEEKS_PER_YEAR

def build_sector_hedged( port_returns, sector_returns, window=ROLL_WINDOW):
    df = pd.concat([port_returns.rename('portfolio'), sector_returns], axis=1).dropna()
    sector_cols = sector_returns.columns
    idx = df.index

    betas = pd.DataFrame(index=idx, columns=sector_cols, dtype=float)

    for i in range(window-1, len(idx)-1):
        window_slice=df.iloc[i-window+1:i+1]

        y = window_slice['portfolio']
        X = window_slice[sector_cols]
        X = sm.add_constant(X)

        model=sm.OLS(y, X).fit()
        betas.iloc[i] = model.params[sector_cols]

    hedge_weights = betas.shift(1)

    hedged = df['portfolio'] - (hedge_weights * df[sector_cols]).sum(axis=1)

    return hedged

carry_return_hedged_LO = build_sector_hedged(carry_return, sector_returns)
carry_return_hedged_LS = build_sector_hedged(carry_return_long_short, sector_returns)

hedged_stats = pd.DataFrame({
    'LO_hedged' : perf_stats(carry_return_hedged_LO),
    'LS_hedged' : perf_stats(carry_return_hedged_LS),
}).T

display(hedged_stats)

,mean_ann,vol_ann,sharpe,skewness,VaR_5,CVaR_5,max_drawdown
LO_hedged,0.114491,0.187061,0.612048,0.276006,-0.032168,-0.057856,-0.360059
LS_hedged,0.001312,0.107468,0.012208,1.075714,-0.021079,-0.030726,-0.198952


In [66]:
#3.2
def linear_decomposition_sector(strategy_ret, sector_returns):
    df = pd.concat([strategy_ret.rename('portfolio'), sector_returns], axis=1).dropna()
    y = df['portfolio']
    X = df.drop(columns='portfolio')
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()
    alpha_ann = model.params['const'] * WEEKS_PER_YEAR
    r2        = model.rsquared
    betas     = model.params.drop('const')
    return alpha_ann, r2, betas, model

alpha_LO_h, r2_LO_h, betas_LO_h, model_LO_h = linear_decomposition_sector(carry_return_hedged_LO, sector_returns)
alpha_LS_h, r2_LS_h, betas_LS_h, model_LS_h = linear_decomposition_sector(carry_return_hedged_LS, sector_returns)

lfd_hedged_stats = pd.DataFrame({
    'alpha_ann': [alpha_LO_h, alpha_LS_h],
    'r_squared': [r2_LO_h,   r2_LS_h],
}, index=['LO_hedged', 'LS_hedged'])

sector_betas_hedged = pd.DataFrame({
    'LO_hedged_beta': betas_LO_h,
    'LS_hedged_beta': betas_LS_h,
})

display(lfd_hedged_stats)
display(sector_betas_hedged)


,alpha_ann,r_squared
LO_hedged,0.124725,0.038215
LS_hedged,-0.007981,0.068981


,LO_hedged_beta,LS_hedged_beta
XLK,0.001554,0.055520
XLI,-0.028150,0.013374
XLF,-0.013703,-0.032341
XLC,-0.011206,-0.023673
XLRE,-0.167448,-0.104127
XLE,0.010218,0.044779
XLY,0.021763,-0.036008
XLB,0.144492,0.022568
XLV,0.007873,0.050750
XLU,-0.077831,-0.082925


***

# 4. Assessing the Forecast

Up to this point we did **not** build an explicit return forecast;
we jumped straight from a signal (dividend yield) to positions.

We could introduce the intermediate step of the return forecasts to try to understand whether the performance is driven by forecasting vs positioning. For illustration, consider the following forecasts:

- **+0.1%** for stocks we go long (top sorted)
- **−0.1%** for stocks we short (bottom sorted).

### 4.1.
- Treat those implied forecasts as a cross-sectional predictor of
  next-period returns.
- Compute an "out-of-sample" $R^2$ for the forecast vs realized
  returns.
- Compute the correlation between the forecast and realized returns.

### 4.2.

In section `3`, we considered a hedged strategies. Repeat `4.1` but this time interpreting our forecasts as pertaining to the hedged residuals of an LFD.

That is, build the forecasts identically, but compare them to the realized $\epsilon_t$ rather than the realized $r_t$. 

Use `SPY` as the single hedging factor. Be careful to estimate $\epsilon_t$ each period with data up to that point in time, rather than using the full sample.

### 4.3.

Repeat `4.2.` but considering all the sectors as the LFD.

In [67]:
#4.1
forecast_step = 0.001

forecast_LO = long_only.astype(float) * forecast_step

forecast_LS = (long_only.astype(float) - short_only.astype(float))*forecast_step

def forecast_stats(forecast, realized):
    mask = forecast.notna() & realized.notna()
    f = forecast[mask]
    r = realized[mask]

    f_vec = f.values.ravel()
    r_vec = r.values.ravel()

    valid = ~np.isnan(f_vec) & ~np.isnan(r_vec)
    f_vec = f_vec[valid]
    r_vec = r_vec[valid]

    ss_res = np.sum((r_vec - f_vec)**2)
    ss_tot = np.sum((r_vec-r_vec.mean())**2)
    R2_oos = 1 - ss_res / ss_tot 

    corr = np.corrcoef(f_vec, r_vec)[0,1]

    return R2_oos, corr

R2_LO, corr_LO = forecast_stats(forecast_LO, ret_fwd)
R2_LS, corr_LS = forecast_stats(forecast_LS, ret_fwd)

print("LO forecast:")
print("  OOS R^2:", R2_LO)
print("  corr(forecast, return):", corr_LO)

print("\nLS forecast:")
print("  OOS R^2:", R2_LS)
print("  corr(forecast, return):", corr_LS)

LO forecast:
  OOS R^2: -0.004524641914186667
  corr(forecast, return): -0.0025653674408033897

LS forecast:
  OOS R^2: -0.005068449178426082
  corr(forecast, return): -0.0008757290545238111


In [71]:
#4.2
def rolling_residuals(returns, factors, window=ROLL_WINDOW):
    idx = returns.index.intersection(factors.index)
    R = returns.loc[idx]
    F = factors.loc[idx]

    eps = pd.DataFrame(index=idx, columns=R.columns, dtype=float)

    factor_cols = F.columns

    for stock in R.columns:
        df_i = pd.concat([R[stock].rename('ret'), F], axis=1).dropna()
        if len(df_i) < window:
            continue

        for t in range(window - 1, len(df_i)):
            w_slice = df_i.iloc[t - window + 1 : t + 1]

            y = w_slice['ret']
            X = sm.add_constant(w_slice[factor_cols])

            model = sm.OLS(y, X).fit()
            # residual at current date
            eps.loc[df_i.index[t], stock] = model.resid.iloc[-1]

    return eps

spy_factor = pd.DataFrame({'SPY': spy_return})

eps_spy = rolling_residuals(ret_fwd, spy_factor)

R2_LO_spy, corr_LO_spy = forecast_stats(forecast_LO, eps_spy)
R2_LS_spy, corr_LS_spy = forecast_stats(forecast_LS, eps_spy)

print("4.2  (SPY-hedged residuals)")
print("LO:  OOS R^2 = {:.4f},  corr = {:.4f}".format(R2_LO_spy, corr_LO_spy))
print("LS:  OOS R^2 = {:.4f},  corr = {:.4f}".format(R2_LS_spy, corr_LS_spy))

4.2  (SPY-hedged residuals)
LO:  OOS R^2 = 0.0002,  corr = 0.0174
LS:  OOS R^2 = 0.0003,  corr = 0.0182


In [74]:
#4.3 
eps_sectors = rolling_residuals(ret_fwd, sector_returns)

R2_LO_sec, corr_LO_sec = forecast_stats(forecast_LO, eps_sectors)
R2_LS_sec, corr_LS_sec = forecast_stats(forecast_LS, eps_sectors)

print("4.3  (sector-hedged residuals)")
print("LO:  OOS R^2 = {:.4f},  corr = {:.4f}".format(R2_LO_sec, corr_LO_sec))
print("LS:  OOS R^2 = {:.4f},  corr = {:.4f}".format(R2_LS_sec, corr_LS_sec))

4.3  (sector-hedged residuals)
LO:  OOS R^2 = -0.0002,  corr = 0.0166
LS:  OOS R^2 = -0.0000,  corr = 0.0175


***

# 5. No-Arbitrage via Cross-Asset Replication

If an asset can be replicated by other assets, an arbitrage-free model should assign consistent forecasts to the asset and its replication portfolio.

### 5.1. LASSO Replication and Forecast Consistency

For a selected asset, use LASSO to find a replication portfolio. Compare the dividend-yield-based forecast for the target asset vs its replication portfolio. 

### 5.2. Arbitrage

Do your forecasts imply arbitrage?

In [82]:
#5.1
from sklearn.linear_model import LassoCV
target_ticker = 'MO'
forecast_step = 0.001 

returns = ret_fwd.copy()
df = returns[[target_ticker]].join(
    returns.drop(columns=[target_ticker]),
    how='inner'
)

df = df.dropna(axis=0).astype(float)

y = df[target_ticker]
X = df.drop(columns=[target_ticker])

lasso = LassoCV(cv=5, fit_intercept=True, random_state=0)
lasso.fit(X,y)

coef = pd.Series(lasso.coef_, index=X.columns)
rep_weights = coef[coef !=0]

rep_return = (returns[rep_weights.index] @ rep_weights).loc[y.index]

forecast_target = forecast_LO[target_ticker].loc[y.index]

forecast_rep = (forecast_LO[rep_weights.index].loc[y.index] @ rep_weights)

comp_forecasts = pd.DataFrame({
    'forecast_target': forecast_target,
    'forecast_rep':    forecast_rep
})

display(comp_forecasts.head())

print("\nCorrelation between forecasts:")
print(comp_forecasts.corr())

print("\nAverage forecast (bps):")
print(10000 * comp_forecasts.mean())

c:\Users\gonza\anaconda3\envs\Portfolio_and_risk_management\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.350e-05, tolerance: 1.493e-05
  model = cd_fast.enet_coordinate_descent(
c:\Users\gonza\anaconda3\envs\Portfolio_and_risk_management\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.925e-05, tolerance: 1.493e-05
  model = cd_fast.enet_coordinate_descent(
c:\Users\gonza\anaconda3\envs\Portfolio_and_risk_management\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations

,forecast_target,forecast_rep
date,,
2020-07-03,0.001,0.000317
2020-07-10,0.001,0.000317
2020-07-17,0.001,0.000317
2020-07-24,0.001,0.000317
2020-07-31,0.001,0.000317



Correlation between forecasts:
                 forecast_target  forecast_rep
forecast_target              NaN           NaN
forecast_rep                 NaN           1.0

Average forecast (bps):
forecast_target    10.000000
forecast_rep        2.921973
dtype: float64


***

# 6. EXTRA - Improvements

### Allocation
Use a different allocation scheme. Above we used equal weightings for any stock that ranked above/below a certain threshold. Other allocations that overweight carry or value could be interesting.

### Signal
Point estimates of the signal may be noisy or misleading. Filter outlier values and take a rolling average of the signal.

### Sorting with Sector Neutrality
Rather than ranking the entire set, rank and allocate within each sector separately, ensuring sector neutrality.

### Momentum
Add a rule that you do not invest in the worst performing stocks over the past year, even if they rank well. And you do not short the best performing stocks over the past year, even if they rank poorly by our signal.